# 14 — Evaluation Metrics and Generalization Practice

This notebook practices classification metrics, regression metrics, threshold sweeps, bootstrap confidence intervals, and basic overfitting diagnostics.

In [ ]:
import numpy as np

## 1. Confusion Matrix Counts

$$
TP, FP, TN, FN
$$

In [ ]:
def confusion_counts(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, fp, tn, fn

## 2. Classification Metrics

$$
Accuracy=\frac{TP+TN}{TP+TN+FP+FN}
$$

$$
Precision=\frac{TP}{TP+FP}
$$

$$
Recall=\frac{TP}{TP+FN}
$$

$$
F1=2\frac{Precision\cdot Recall}{Precision+Recall}
$$

In [ ]:
def classification_metrics(y_true, y_pred):
    tp, fp, tn, fn = confusion_counts(y_true, y_pred)
    accuracy = (tp + tn) / (tp + fp + tn + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return accuracy, precision, recall, f1


y_true = np.array([0] * 90 + [1] * 10)
y_pred_all_negative = np.array([0] * 100)
y_pred_better = np.array([0] * 85 + [1] * 5 + [1] * 7 + [0] * 3)

classification_metrics(y_true, y_pred_all_negative), classification_metrics(y_true, y_pred_better)

## 3. Threshold Sweep

In [ ]:
probabilities = np.concatenate([
    np.linspace(0.01, 0.45, 90),
    np.linspace(0.35, 0.95, 10),
])

thresholds = np.linspace(0.2, 0.8, 7)

for threshold in thresholds:
    y_pred = (probabilities >= threshold).astype(int)
    accuracy, precision, recall, f1 = classification_metrics(y_true, y_pred)
    print(threshold, precision, recall, f1)

## 4. Regression Metrics

$$
MAE=\frac{1}{n}\sum_i |y_i-\hat{y}_i|
$$

$$
RMSE=\sqrt{\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2}
$$

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot


y_reg_true = np.array([10, 12, 15, 18, 20, 24], dtype=float)
y_reg_pred = np.array([9, 13, 14, 17, 23, 22], dtype=float)

mae(y_reg_true, y_reg_pred), mse(y_reg_true, y_reg_pred), rmse(y_reg_true, y_reg_pred), r2_score(y_reg_true, y_reg_pred)

## 5. Bootstrap Confidence Interval for Accuracy

In [ ]:
def bootstrap_accuracy_ci(y_true, y_pred, n_boot=3000, confidence=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        scores.append(np.mean(y_true[idx] == y_pred[idx]))
    alpha = 1 - confidence
    lower = np.percentile(scores, 100 * alpha / 2)
    upper = np.percentile(scores, 100 * (1 - alpha / 2))
    return lower, upper

bootstrap_accuracy_ci(y_true, y_pred_better)

## 6. Overfitting Diagnosis from Loss Curves

In [ ]:
epochs = np.arange(1, 61)
train_losses = 2.0 * np.exp(-epochs / 25) + 0.1
val_losses = 1.5 * np.exp(-epochs / 22) + 0.25 + 0.01 * np.maximum(epochs - 30, 0)

best_val_epoch = np.argmin(val_losses) + 1
best_val_epoch, train_losses[-1], val_losses[-1]

## Reflection

Evaluation is not one number. It is the process of checking whether a model generalizes, which mistakes it makes, and whether the chosen metric matches the real problem.